# REGPLEX_Local — REGPLEX v10
Publication-oriented local notebook for hierarchical perplexity ensemble analysis.


## 1. Scientific Introduction
REGPLEX v10 uses mono/di/tri information-theoretic observers to detect regulatory valleys.


## 2. Algorithm Overview
The pipeline computes layer perplexities once, builds multi-scale LPC profiles, and ensembles consensus signals.


## 3. Loading FASTA
Load FASTA and inspect sequence metadata before analysis.


## 4. Mono Perplexity
Compute mononucleotide perplexity profile.


## 5. Di Perplexity
Compute dinucleotide perplexity profile (primary layer).


## 6. Tri Perplexity
Compute trinucleotide perplexity profile.


## 7. Multi-scale Landscapes
Generate rolling landscapes for each scale and layer.


## 8. Three-window Contrast
Compute upstream/candidate/downstream LPC per scale.


## 9. Consensus Construction
Create per-layer and final ensemble ConsensusLPC.


## 10. Kadane Optimization
Detect consensus valley core with bounded Kadane.


## 11. Valley Expansion
Expand boundaries while ConsensusLPC remains positive.


## 12. Valley Merging
Merge overlapping or close valleys and keep explainability metrics.


## 13. Visualization
Generate publication-ready figures from the visualization module.


## 14. Motif Annotation
Annotate detected valleys with regex/IUPAC motifs.


## 15. Export Results
Export tabular outputs for downstream analysis.


In [ ]:
import pandas as pd
from regplex_core import (
    parse_fasta,
    analyze_sequence,
    domains_dataframe,
    compute_mono_perplexity,
    compute_di_perplexity,
    compute_tri_perplexity,
)
from motif_engine import compile_motifs, annotate_domains
from visualization import (
    plot_perplexity_layers,
    plot_layer_consensus,
    plot_consensus_lpc,
    plot_domain_ranking,
    plot_motif_architecture,
)


In [ ]:
with open('examples/ecoli.fasta', encoding='utf-8') as fh:
    records = parse_fasta(fh.read())
header, seq = records[0]
print(header, len(seq))


In [ ]:
mono = compute_mono_perplexity(seq)
di = compute_di_perplexity(seq)
tri = compute_tri_perplexity(seq)
len(mono), len(di), len(tri)


In [ ]:
result = analyze_sequence(
    header,
    seq,
    scales=[25, 50, 100, 200, 400],
    landscape_method='mean',
    normalization_method='robust_z',
    ensemble_method='median',
)
df = domains_dataframe([result])
df.head()


In [ ]:
plot_perplexity_layers(result.mono, result.di, result.tri).show()
plot_layer_consensus(result.layer_consensus, result.domains).show()
plot_consensus_lpc(result.consensus_lpc, result.domains).show()
plot_domain_ranking(result.domains).show()


In [ ]:
motifs = compile_motifs('TATAWAWR
GGGN{1,7}GGG')
annotate_domains(result.domains, motifs)
plot_motif_architecture(result.domains).show()
domains_dataframe([result]).head()


In [ ]:
out = domains_dataframe([result])
out.to_csv('examples/sample_output.csv', index=False)
out[['ID','Start','End','ValleyScore','OverallSupport']].head()
